# Genetic Algorithms - Tutorial 2: Advanced Operators and Techniques

Welcome to the second tutorial on **Genetic Algorithms**! This tutorial builds upon Tutorial 1 and covers advanced techniques used in modern genetic algorithms.

## What You'll Learn:

1. **Advanced Selection Methods** - Rank-based, SUS, Boltzmann selection
2. **Advanced Crossover Operators** - Arithmetic, BLX-α, SBX
3. **Advanced Mutation Strategies** - Polynomial, adaptive, self-adaptive
4. **Elitism and Replacement** - Preserving best solutions
5. **Constraint Handling** - Dealing with problem constraints
6. **Diversity Maintenance** - Preventing premature convergence
7. **Adaptive Parameters** - Dynamic parameter adjustment
8. **Performance Comparison** - Benchmarking different techniques

**Prerequisites:** Complete Tutorial 1 first!

---

## Table of Contents

1. [Setup and Review](#1)
2. [Advanced Selection Methods](#2)
3. [Advanced Crossover Operators](#3)
4. [Advanced Mutation Strategies](#4)
5. [Elitism and Replacement Strategies](#5)
6. [Constraint Handling](#6)
7. [Diversity Maintenance](#7)
8. [Adaptive Parameters](#8)
9. [Complete Advanced GA](#9)
10. [Performance Comparison](#10)
11. [Conclusion](#11)

---

<a name='1'></a>
## 1 - Setup and Review

Let's import all necessary libraries and review basic concepts from Tutorial 1.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import sys
import time

# Import basic utilities from Tutorial 1
sys.path.append('../GA_Tutorial_1_Basics')
from ga_utils_basics import (
    initialize_population_real,
    evaluate_fitness,
    sphere_function,
    rastrigin_function,
    rosenbrock_function,
    plot_fitness_evolution,
    calculate_diversity
)

# Import advanced utilities
from ga_utils_intermediate import *
from public_tests import *

# Set random seed
np.random.seed(42)

# Configure matplotlib
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ All imports successful!")
print("✓ Ready to learn advanced GA techniques!")

### Quick Review from Tutorial 1

**Basic GA Components:**
- Population initialization
- Fitness evaluation
- Selection (roulette wheel, tournament)
- Crossover (single-point, two-point, uniform)
- Mutation (bit-flip, Gaussian)

**Why Advanced Techniques?**

Basic GAs work well for simple problems, but advanced techniques provide:
- ✅ Better convergence on difficult problems
- ✅ More robust performance
- ✅ Ability to handle constraints
- ✅ Prevention of premature convergence
- ✅ Better exploration-exploitation balance

---

<a name='2'></a>
## 2 - Advanced Selection Methods

Selection determines which individuals reproduce. Advanced methods offer better control over selection pressure.

### 2.1 Rank-Based Selection

**Problem with Fitness-Proportionate Selection:**
- If one individual has fitness 1000 and others have fitness 10, it dominates
- Susceptible to scaling issues

**Rank-Based Solution:**
- Selection probability based on **rank**, not raw fitness
- More robust to fitness scaling

**Formula:**
$$P(i) = \frac{2 - SP + 2(SP-1)\frac{rank_i - 1}{N-1}}{N}$$

Where:
- $SP$ = selection pressure (1.0-2.0)
- $N$ = population size
- $rank_i$ = rank of individual i (0 = worst, N-1 = best)

In [ ]:
# Test rank-based selection
print("="*70)
print("Rank-Based Selection Demo")
print("="*70)

# Create population with extreme fitness differences
test_pop = np.array([
    [1.0, 1.0],
    [2.0, 2.0],
    [3.0, 3.0],
    [4.0, 4.0],
    [5.0, 5.0]
])

# Extreme fitness values (one dominates)
extreme_fitness = np.array([1.0, 2.0, 3.0, 5.0, 1000.0])

print("\nPopulation fitness (one super-fit individual):")
for i, (ind, fit) in enumerate(zip(test_pop, extreme_fitness)):
    print(f"Individual {i}: {ind} - Fitness: {fit:.1f}")

# Compare selection methods
print("\n" + "-"*70)
print("Selection with Rank-Based (robust to extreme fitness)")
print("-"*70)

np.random.seed(42)
selected_rank = rank_based_selection(test_pop, extreme_fitness, num_parents=10, selection_pressure=1.5)

# Count selections
selection_count = np.zeros(len(test_pop))
for selected in selected_rank:
    for i, ind in enumerate(test_pop):
        if np.array_equal(selected, ind):
            selection_count[i] += 1
            break

print("\nSelection frequency (out of 10):")
for i, count in enumerate(selection_count):
    print(f"Individual {i} (fitness={extreme_fitness[i]:7.1f}): {int(count)} times")

# Run test
test_rank_based_selection()

### 2.2 Stochastic Universal Sampling (SUS)

**Problem with Roulette Wheel:**
- High variance - lucky/unlucky individuals
- Multiple spins can miss good individuals

**SUS Solution:**
- Single spin with multiple evenly-spaced pointers
- Lower variance, more representative sampling

**Visual:**
```
Roulette Wheel:  [spin] [spin] [spin] [spin]  (4 random spins)
SUS:             [  equally spaced pointers  ]  (1 random start)
```

In [ ]:
# Compare SUS vs Roulette Wheel
print("="*70)
print("Stochastic Universal Sampling (SUS) Demo")
print("="*70)

# Create test population
test_fitness = np.array([10.0, 20.0, 30.0, 40.0])
test_pop = np.eye(4)  # Easy to track

print("\nPopulation fitness:")
for i, fit in enumerate(test_fitness):
    print(f"Individual {i}: Fitness = {fit:.1f}")

# Multiple runs to see variance
print("\n" + "-"*70)
print("Running 5 trials to compare variance...")
print("-"*70)

sus_results = []
for trial in range(5):
    selected = stochastic_universal_sampling(test_pop, test_fitness, num_parents=8)
    counts = np.zeros(4)
    for s in selected:
        idx = np.argmax(s)
        counts[idx] += 1
    sus_results.append(counts)

print("\nSUS Selection Counts (8 parents selected):")
print("Expected proportions: [1, 2, 3, 4] / 10 * 8 = [0.8, 1.6, 2.4, 3.2]")
print()
for i, counts in enumerate(sus_results):
    print(f"Trial {i+1}: {counts}")

print(f"\nAverage: {np.mean(sus_results, axis=0)}")
print(f"Std Dev: {np.std(sus_results, axis=0):.3f}")

# Run test
test_stochastic_universal_sampling()

### 2.3 Boltzmann Selection

**Inspired by:** Simulated Annealing

**Key Idea:** Selection pressure controlled by temperature

$$P(i) \propto e^{\frac{fitness_i}{T}}$$

Where $T$ = temperature:
- **High T**: Almost random selection (exploration)
- **Low T**: Strong selection pressure (exploitation)
- **Decreasing T**: Start exploratory, become selective

**Use Case:** Adaptive selection pressure over time

In [ ]:
# Demonstrate Boltzmann selection with different temperatures
print("="*70)
print("Boltzmann Selection with Temperature Control")
print("="*70)

test_fitness = np.array([10.0, 20.0, 30.0, 40.0, 50.0])
test_pop = np.eye(5)

print("\nPopulation fitness:")
for i, fit in enumerate(test_fitness):
    print(f"Individual {i}: Fitness = {fit:.1f}")

temperatures = [10.0, 5.0, 1.0, 0.5]

print("\n" + "-"*70)
print("Testing different temperatures (20 selections each)")
print("-"*70)

for temp in temperatures:
    np.random.seed(42)
    counts = np.zeros(5)
    
    for _ in range(20):
        selected = boltzmann_selection(test_pop, test_fitness, 1, temperature=temp)
        idx = np.argmax(selected[0])
        counts[idx] += 1
    
    print(f"\nT = {temp:5.1f}: {counts.astype(int)} | ", end="")
    if temp >= 5.0:
        print("High T → More random (exploration)")
    else:
        print("Low T → Strong selection (exploitation)")

test_boltzmann_selection()

**Observations:**
- Rank-based: Robust to fitness scaling
- SUS: Lower variance than roulette wheel
- Boltzmann: Adaptive selection pressure via temperature

---

<a name='3'></a>
## 3 - Advanced Crossover Operators

For **real-valued** optimization, specialized crossover operators often outperform binary-inspired methods.

### 3.1 Arithmetic Crossover

**Simplest real-valued crossover:**

$$offspring_1 = \alpha \cdot parent_1 + (1-\alpha) \cdot parent_2$$
$$offspring_2 = (1-\alpha) \cdot parent_1 + \alpha \cdot parent_2$$

Where $\alpha \in [0, 1]$ (typically 0.5)

**Properties:**
- Offspring are convex combinations of parents
- Always stay within parent bounds
- Conservative (limited exploration)

In [ ]:
# Test arithmetic crossover
print("="*70)
print("Arithmetic Crossover")
print("="*70)

parent1 = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
parent2 = np.array([5.0, 4.0, 3.0, 2.0, 1.0])

print(f"\nParent 1: {parent1}")
print(f"Parent 2: {parent2}")

for alpha in [0.3, 0.5, 0.7]:
    off1, off2 = arithmetic_crossover(parent1, parent2, alpha=alpha)
    print(f"\nα = {alpha}:")
    print(f"  Offspring 1: {off1}")
    print(f"  Offspring 2: {off2}")

test_arithmetic_crossover()

### 3.2 BLX-α Crossover (Blend Crossover)

**More exploratory** than arithmetic crossover.

For each gene:
1. Find range: $[min(p_1, p_2), max(p_1, p_2)]$
2. Extend by factor $\alpha$:
   - $lower = min - \alpha \cdot range$
   - $upper = max + \alpha \cdot range$
3. Sample uniformly: $offspring \sim U(lower, upper)$

**Common:** $\alpha = 0.5$ (extends 50% beyond parent range)

**Properties:**
- More exploration than arithmetic
- Can create offspring outside parent range
- Popular in continuous optimization

In [ ]:
# Test BLX-alpha crossover
print("="*70)
print("BLX-α Crossover")
print("="*70)

parent1 = np.array([2.0, 2.0])
parent2 = np.array([4.0, 4.0])
bounds = (-10.0, 10.0)

print(f"\nParent 1: {parent1}")
print(f"Parent 2: {parent2}")
print(f"Range per gene: {parent2 - parent1}")

print("\n" + "-"*70)
print("Multiple offspring with α=0.5 (extension = 50% of range)")
print("-"*70)

np.random.seed(42)
print("\nExpected range per gene: [2.0 - 0.5*2, 4.0 + 0.5*2] = [1.0, 5.0]\n")

for i in range(5):
    off1, off2 = blx_alpha_crossover(parent1, parent2, alpha=0.5, bounds=bounds)
    print(f"Trial {i+1}:")
    print(f"  Offspring 1: {off1}")
    print(f"  Offspring 2: {off2}")

test_blx_alpha_crossover()

### 3.3 Simulated Binary Crossover (SBX)

**Used in NSGA-II** and many modern GAs.

**Key Idea:** Mimic the behavior of single-point binary crossover for real values.

**Distribution parameter** $\eta_c$ (eta):
- **Large $\eta_c$ (e.g., 20)**: Offspring near parents (exploitation)
- **Small $\eta_c$ (e.g., 2)**: Offspring spread out (exploration)

**Properties:**
- Self-adaptive spread
- Widely used in multi-objective optimization
- Good balance of exploration/exploitation

In [ ]:
# Test SBX with different eta values
print("="*70)
print("Simulated Binary Crossover (SBX)")
print("="*70)

parent1 = np.array([2.0, 2.0])
parent2 = np.array([4.0, 4.0])
bounds = (-10.0, 10.0)

print(f"\nParent 1: {parent1}")
print(f"Parent 2: {parent2}")

eta_values = [2, 10, 20]

for eta in eta_values:
    print(f"\n" + "-"*70)
    print(f"η = {eta} ({'low' if eta < 5 else 'high'} - {'exploration' if eta < 5 else 'exploitation'})")
    print("-"*70)
    
    np.random.seed(42)
    for i in range(3):
        off1, off2 = simulated_binary_crossover(parent1, parent2, eta=eta, bounds=bounds)
        print(f"Trial {i+1}: Offspring 1 = {off1}, Offspring 2 = {off2}")

test_simulated_binary_crossover()

**Key Observations:**
- Arithmetic: Conservative, stays between parents
- BLX-α: Exploratory, extends beyond parents
- SBX: Adaptive, controlled by η parameter

---

<a name='4'></a>
## 4 - Advanced Mutation Strategies

### 4.1 Polynomial Mutation

**Companion to SBX**, used in NSGA-II.

**Properties:**
- Probability of small changes > large changes
- Controlled by distribution index $\eta_m$
- Self-adaptive perturbation

**Parameter $\eta_m$:**
- Large: Small mutations (fine-tuning)
- Small: Large mutations (exploration)

In [ ]:
# Test polynomial mutation
print("="*70)
print("Polynomial Mutation")
print("="*70)

chromosome = np.array([0.0, 0.0, 0.0, 0.0, 0.0])
mutation_rate = 0.5
bounds = (-5.0, 5.0)

print(f"\nOriginal: {chromosome}")
print(f"Mutation rate: {mutation_rate}")
print(f"Bounds: {bounds}")

eta_values = [5, 20, 50]

for eta in eta_values:
    print(f"\n" + "-"*70)
    print(f"η_m = {eta} ({'large mutations' if eta < 10 else 'small mutations'})")
    print("-"*70)
    
    np.random.seed(42)
    for i in range(3):
        mutated = polynomial_mutation(chromosome, mutation_rate, bounds, eta=eta)
        print(f"Trial {i+1}: {mutated}")

test_polynomial_mutation()

### 4.2 Adaptive Mutation

**Problem:** Fixed mutation rate may not be optimal throughout evolution.

**Solution:** Decrease mutation over time:
- **Early generations**: High mutation (exploration)
- **Late generations**: Low mutation (fine-tuning)

$$mutation\_rate(t) = rate_{initial} - (rate_{initial} - rate_{final}) \cdot \frac{t}{T}$$

Where:
- $t$ = current generation
- $T$ = max generations

In [ ]:
# Demonstrate adaptive mutation
print("="*70)
print("Adaptive Mutation Rate Over Generations")
print("="*70)

chromosome = np.array([0.0, 0.0, 0.0])
bounds = (-5.0, 5.0)
max_generations = 100

generations_to_test = [0, 25, 50, 75, 99]

print("\nMutation strength at different generations:")
print(f"Initial rate: 0.1, Final rate: 0.01\n")

for gen in generations_to_test:
    print(f"\nGeneration {gen}/{max_generations}:")
    
    np.random.seed(42)
    mutations = []
    for _ in range(5):
        mutated = adaptive_mutation(chromosome, gen, max_generations, bounds)
        mutations.append(mutated)
    
    avg_change = np.mean([np.abs(m - chromosome).sum() for m in mutations])
    print(f"  Average total change: {avg_change:.4f}")
    print(f"  Example mutated: {mutations[0]}")

test_adaptive_mutation()

### 4.3 Self-Adaptive Mutation

**Most advanced:** Mutation parameters **evolve with the solution**!

**Concept:** Each individual carries:
1. Solution vector $\mathbf{x}$
2. Strategy parameters $\mathbf{\sigma}$ (mutation strengths)

**Both evolve together:**
- Good strategies → better solutions → get selected → propagate
- Poor strategies → worse solutions → eliminated

**Used in:** Evolution Strategies (ES), CMA-ES

In [ ]:
# Demonstrate self-adaptive mutation
print("="*70)
print("Self-Adaptive Mutation")
print("="*70)

chromosome = np.array([0.0, 0.0, 0.0])
strategy_params = np.array([0.5, 0.5, 0.5])  # Initial mutation strengths
bounds = (-5.0, 5.0)

print(f"\nOriginal chromosome: {chromosome}")
print(f"Initial strategy parameters: {strategy_params}")

print("\n" + "-"*70)
print("Evolving both solution AND strategy parameters")
print("-"*70)

np.random.seed(42)

current_chrom = chromosome.copy()
current_strategy = strategy_params.copy()

for i in range(5):
    current_chrom, current_strategy = self_adaptive_mutation(
        current_chrom, current_strategy, bounds, tau=0.1, tau_prime=0.05
    )
    print(f"\nIteration {i+1}:")
    print(f"  Chromosome: {current_chrom}")
    print(f"  Strategy:   {current_strategy}")
    print(f"  (Strategy params evolve to match problem!)

test_self_adaptive_mutation()

**Mutation Strategy Summary:**

| Method | Control | Complexity | Best For |
|--------|---------|------------|----------|
| Gaussian | Fixed rate | Low | Simple problems |
| Polynomial | Distribution index | Medium | General purpose |
| Adaptive | Time-based | Medium | Known duration |
| Self-Adaptive | Evolves automatically | High | Complex landscapes |

---

<a name='5'></a>
## 5 - Elitism and Replacement Strategies

### 5.1 The Need for Elitism

**Problem:** Without elitism, best solution can be lost!

**Example:**
```
Generation 10: Best fitness = 100 ✓
Generation 11: Best fitness = 95  ✗ (regression!)
```

**Elitism:** Always preserve best individuals

**Benefits:**
- ✅ Monotonic improvement (best never gets worse)
- ✅ Faster convergence
- ✅ Guaranteed to keep good solutions

**Drawback:**
- ⚠️ Can reduce diversity if elite size too large

In [ ]:
# Demonstrate elitism
print("="*70)
print("Elitism Demonstration")
print("="*70)

# Simulate two scenarios
np.random.seed(42)

# Old generation (has some good solutions)
old_pop = initialize_population_real(10, 2, (-5, 5))
old_fitness = np.random.uniform(50, 100, 10)
old_fitness[0] = 150  # Best solution
old_fitness[1] = 140  # Second best

# New generation (random, may not have good solutions)
new_pop = initialize_population_real(10, 2, (-5, 5))
new_fitness = np.random.uniform(40, 90, 10)

print("\nOld generation:")
print(f"  Best fitness: {np.max(old_fitness):.2f}")
print(f"  All fitness: {np.sort(old_fitness)[::-1]}")

print("\nNew generation (without elitism):")
print(f"  Best fitness: {np.max(new_fitness):.2f}")
print(f"  All fitness: {np.sort(new_fitness)[::-1]}")
print("  ⚠️ LOST BEST SOLUTION!")

# Apply elitism
elite_size = 2
new_pop_elite, new_fitness_elite = elitist_replacement(
    old_pop, old_fitness, new_pop.copy(), new_fitness.copy(), elite_size
)

print(f"\nNew generation (WITH {elite_size} elites):")
print(f"  Best fitness: {np.max(new_fitness_elite):.2f}")
print(f"  All fitness: {np.sort(new_fitness_elite)[::-1]}")
print("  ✓ BEST SOLUTIONS PRESERVED!")

test_elitist_replacement()

### 5.2 Steady-State Replacement

**Alternative to generational replacement:**

**Generational GA:**
- Create full new population
- Replace entire old population

**Steady-State GA:**
- Generate 1-2 offspring at a time
- Replace worst individual if offspring better
- More gradual evolution

**Advantages:**
- Implicit elitism
- Smoother convergence
- Can update incrementally

In [ ]:
# Demonstrate steady-state replacement
print("="*70)
print("Steady-State Replacement")
print("="*70)

# Small population
pop = initialize_population_real(5, 2, (-5, 5))
fitness = np.array([10.0, 20.0, 30.0, 40.0, 50.0])

print("\nInitial population fitness:")
print(fitness)
print(f"Worst individual: index {np.argmin(fitness)}, fitness = {np.min(fitness)}")

# Generate good offspring
good_offspring = np.array([2.5, 2.5])
good_fitness = 45.0

print(f"\nNew offspring fitness: {good_fitness}")
print("Action: Replace worst (10.0) with offspring (45.0)")

pop, fitness = steady_state_replacement(pop, fitness, good_offspring, good_fitness)

print("\nUpdated population fitness:")
print(fitness)
print("✓ Worst replaced, population improved!")

# Generate weak offspring
weak_offspring = np.array([1.0, 1.0])
weak_fitness = 5.0

print(f"\n" + "-"*70)
print(f"New offspring fitness: {weak_fitness}")
print("Action: Don't replace (offspring worse than worst)")

pop_before = fitness.copy()
pop, fitness = steady_state_replacement(pop, fitness, weak_offspring, weak_fitness)

print("\nPopulation fitness (unchanged):")
print(fitness)
print("✓ Weak offspring rejected!")

test_steady_state_replacement()

---

<a name='6'></a>
## 6 - Constraint Handling

Real-world problems often have **constraints**:
- Budget limits
- Physical constraints
- Regulatory requirements

**Example Problem:**
$$
\begin{align}
\text{Minimize: } & f(x) = x_1^2 + x_2^2 \\
\text{Subject to: } & x_1 + x_2 \geq 2 \\
& x_1, x_2 \in [-5, 5]
\end{align}
$$

### 6.1 Penalty Function Method

**Add penalty for constraint violations:**

$$fitness_{penalized} = fitness + penalty \cdot \sum violations$$

**Tuning:** Penalty coefficient balances objectives vs constraints

In [ ]:
# Demonstrate penalty function
print("="*70)
print("Constraint Handling with Penalty Function")
print("="*70)

def constrained_problem(x):
    """Minimize x1^2 + x2^2 subject to x1 + x2 >= 2"""
    objective = x[0]**2 + x[1]**2
    
    # Constraint: x1 + x2 >= 2, violation if < 2
    constraint = 2 - (x[0] + x[1])  # Positive if violated
    
    return objective, [constraint]

# Test solutions
solutions = [
    np.array([0.0, 0.0]),  # Infeasible (violates constraint)
    np.array([1.0, 1.0]),  # Feasible (satisfies constraint)
    np.array([2.0, 0.0]),  # Feasible (on boundary)
]

penalty_coef = 1000

print(f"\nConstraint: x1 + x2 >= 2")
print(f"Penalty coefficient: {penalty_coef}\n")

for i, sol in enumerate(solutions):
    obj, constraints = constrained_problem(sol)
    penalized = penalty_function(obj, constraints, penalty_coef)
    
    print(f"Solution {i+1}: {sol}")
    print(f"  Sum: {sol[0] + sol[1]:.2f}")
    print(f"  Objective: {obj:.2f}")
    print(f"  Constraint violation: {max(0, constraints[0]):.2f}")
    print(f"  Penalized objective: {penalized:.2f}")
    print(f"  Status: {'INFEASIBLE ✗' if constraints[0] > 0 else 'Feasible ✓'}")
    print()

test_penalty_function()

### 6.2 Death Penalty

**Simplest method:** Reject all infeasible solutions.

**Pros:**
- ✅ Simple to implement
- ✅ Always returns feasible solution

**Cons:**
- ❌ May reject too many solutions
- ❌ Can fail if feasible region is small

### 6.3 Repair Mechanisms

**Fix infeasible solutions** by mapping to feasible region.

**Example:** Clip to bounds

```python
if x[i] < lower_bound:
    x[i] = lower_bound
if x[i] > upper_bound:
    x[i] = upper_bound
```

In [ ]:
# Demonstrate repair mechanism
print("="*70)
print("Repair Mechanism for Bound Constraints")
print("="*70)

infeasible = np.array([-10.0, 7.0, 3.0, -2.0, 15.0])
bounds = (-5.0, 5.0)

print(f"\nInfeasible solution: {infeasible}")
print(f"Bounds: {bounds}")

repaired = repair_bounds(infeasible, bounds)

print(f"\nRepaired solution: {repaired}")
print("\nChanges:")
for i, (orig, rep) in enumerate(zip(infeasible, repaired)):
    if orig != rep:
        print(f"  Gene {i}: {orig:.1f} → {rep:.1f} {'(clipped to lower)' if rep == bounds[0] else '(clipped to upper)'}")

test_repair_bounds()

---

<a name='7'></a>
## 7 - Diversity Maintenance

**Problem:** Premature convergence
- Population becomes too similar
- Gets stuck in local optimum
- No exploration of search space

**Solution:** Maintain diversity!

### 7.1 Crowding Distance

Used in **NSGA-II** for multi-objective optimization.

**Idea:** Prefer individuals in less crowded regions

**Calculation:**
- Sort population by fitness
- Boundary individuals get infinite distance
- Others: distance = fitness gap to neighbors

In [ ]:
# Demonstrate crowding distance
print("="*70)
print("Crowding Distance Calculation")
print("="*70)

# Create population with varying spread
test_pop = np.array([
    [1.0],
    [2.0],
    [2.1],  # Crowded
    [2.2],  # Crowded
    [5.0]
])

test_fitness = np.array([10.0, 20.0, 21.0, 22.0, 50.0])

distances = crowding_distance(test_pop, test_fitness)

print("\nPopulation fitness and crowding distances:")
print(f"{'Individual':<12} {'Fitness':<10} {'Crowding Distance':<20} {'Status'}")
print("-" * 70)

for i, (fit, dist) in enumerate(zip(test_fitness, distances)):
    status = "Boundary" if np.isinf(dist) else ("Isolated" if dist > 0.1 else "Crowded")
    dist_str = "∞" if np.isinf(dist) else f"{dist:.4f}"
    print(f"{i:<12} {fit:<10.1f} {dist_str:<20} {status}")

print("\n💡 Higher crowding distance = more isolated = preserve for diversity")

test_crowding_distance()

### 7.2 Niching with Fitness Sharing

**Idea:** Penalize individuals in crowded areas

**Shared Fitness:**
$$fitness_{shared}(i) = \frac{fitness(i)}{\sum_j sh(d_{ij})}$$

Where:
- $d_{ij}$ = distance between individuals i and j
- $sh(d)$ = sharing function (decreases with distance)

**Result:** Multiple niches (sub-populations) can coexist

In [ ]:
# Demonstrate niching
print("="*70)
print("Niching with Fitness Sharing")
print("="*70)

# Population with two clusters
cluster1 = np.array([[0.0, 0.0], [0.1, 0.1], [0.2, 0.2]])  # Around origin
cluster2 = np.array([[5.0, 5.0], [5.1, 5.1], [5.2, 5.2]])  # Around (5,5)
test_pop = np.vstack([cluster1, cluster2])

# All have same raw fitness
test_fitness = np.array([100.0, 100.0, 100.0, 100.0, 100.0, 100.0])

print("\nPopulation (2 clusters, all equal fitness):")
for i, (ind, fit) in enumerate(zip(test_pop, test_fitness)):
    print(f"Individual {i}: {ind} - Fitness: {fit:.1f}")

# Apply niching with appropriate sigma
sigma_share = 1.0

print(f"\n" + "-"*70)
print(f"Applying niching selection (σ_share = {sigma_share})")
print("-"*70)

np.random.seed(42)
selected = niching_selection(test_pop, test_fitness, num_parents=6, sigma_share=sigma_share)

# Count selections from each cluster
count_cluster1 = 0
count_cluster2 = 0

for sel in selected:
    if np.linalg.norm(sel - np.array([0.0, 0.0])) < 2.0:
        count_cluster1 += 1
    else:
        count_cluster2 += 1

print(f"\nSelected from Cluster 1 (near origin): {count_cluster1}")
print(f"Selected from Cluster 2 (near 5,5):     {count_cluster2}")
print("\n💡 Niching promotes selection from both clusters (diversity!)")

test_niching_selection()

---

<a name='8'></a>
## 8 - Adaptive Parameters

**Fixed parameters** may be suboptimal. **Adaptive parameters** adjust during evolution.

### 8.1 Adaptive Crossover Rate

In [ ]:
# Demonstrate adaptive crossover rate
print("="*70)
print("Adaptive Crossover Rate")
print("="*70)

max_gen = 100
initial_rate = 0.9
final_rate = 0.6

generations = [0, 25, 50, 75, 99]
rates = [adaptive_crossover_rate(g, max_gen, initial_rate, final_rate) for g in generations]

print(f"\nCrossover rate schedule (start={initial_rate}, end={final_rate}):")
print()
for gen, rate in zip(generations, rates):
    print(f"Generation {gen:3d}: Crossover rate = {rate:.3f}")

print("\n💡 High crossover early (exploration), lower later (exploitation)")

# Visualize
all_gens = np.arange(max_gen)
all_rates = [adaptive_crossover_rate(g, max_gen, initial_rate, final_rate) for g in all_gens]

plt.figure(figsize=(10, 4))
plt.plot(all_gens, all_rates, linewidth=2, color='blue')
plt.xlabel('Generation')
plt.ylabel('Crossover Rate')
plt.title('Adaptive Crossover Rate Over Generations')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

test_adaptive_crossover_rate()

### 8.2 Fitness-Based Adaptive Mutation

**Idea:** Adapt mutation rate based on individual fitness

- **Good solutions** (high fitness): Low mutation (preserve)
- **Poor solutions** (low fitness): High mutation (explore)

$$mutation\_rate(i) = 0.5 \cdot \left(1 - \frac{fitness_i - fitness_{min}}{fitness_{max} - fitness_{min}}\right)^k$$

In [ ]:
# Demonstrate fitness-based adaptive mutation
print("="*70)
print("Fitness-Based Adaptive Mutation Rate")
print("="*70)

fitness_values = np.array([10.0, 30.0, 50.0, 70.0, 90.0])
max_fit = np.max(fitness_values)
min_fit = np.min(fitness_values)

print("\nPopulation fitness and corresponding mutation rates:")
print(f"{'Individual':<12} {'Fitness':<10} {'Mutation Rate':<15} {'Strategy'}")
print("-" * 70)

for i, fit in enumerate(fitness_values):
    mut_rate = adaptive_mutation_rate_fitness(fit, max_fit, min_fit, k=2)
    strategy = "High mutation" if mut_rate > 0.3 else ("Medium mutation" if mut_rate > 0.1 else "Low mutation")
    print(f"{i:<12} {fit:<10.1f} {mut_rate:<15.4f} {strategy}")

print("\n💡 Poor individuals mutate more (exploration)")
print("💡 Good individuals mutate less (exploitation)")

test_adaptive_mutation_rate_fitness()

---

<a name='9'></a>
## 9 - Complete Advanced GA

Now let's build a complete GA using advanced techniques!

### YOUR TASK: Implement Advanced GA

Combine all the techniques you've learned.

In [ ]:
def advanced_genetic_algorithm(fitness_function, num_variables, bounds,
                               pop_size=100, max_generations=200,
                               elite_size=2,
                               selection_method='rank',
                               crossover_method='sbx',
                               mutation_method='polynomial',
                               eta_crossover=20,
                               eta_mutation=20,
                               initial_mutation_rate=0.1,
                               verbose=True):
    """
    Advanced genetic algorithm with modern techniques.
    
    Arguments:
    fitness_function -- function to optimize
    num_variables -- number of variables
    bounds -- tuple (lower, upper)
    pop_size -- population size
    max_generations -- max generations
    elite_size -- number of elites to preserve
    selection_method -- 'rank', 'sus', 'boltzmann'
    crossover_method -- 'sbx', 'blx', 'arithmetic'
    mutation_method -- 'polynomial', 'adaptive'
    eta_crossover -- distribution index for SBX
    eta_mutation -- distribution index for polynomial mutation
    initial_mutation_rate -- starting mutation rate
    verbose -- print progress
    
    Returns:
    best_solution, best_fitness, history
    """
    
    # Initialize population
    population = initialize_population_real(pop_size, num_variables, bounds)
    
    # History
    history = {
        'best': [],
        'average': [],
        'worst': [],
        'diversity': []
    }
    
    # Main evolution loop
    for generation in range(max_generations):
        
        # Evaluate fitness
        fitness_values = evaluate_fitness(population, fitness_function,
                                         num_variables, bounds, bits_per_variable=None,
                                         encoding='real')
        
        # Track statistics
        best_idx = np.argmax(fitness_values)
        best_individual = population[best_idx]
        
        history['best'].append(np.max(fitness_values))
        history['average'].append(np.mean(fitness_values))
        history['worst'].append(np.min(fitness_values))
        history['diversity'].append(calculate_diversity(population))
        
        # Print progress
        if verbose and (generation % 40 == 0 or generation == max_generations - 1):
            print(f"\nGen {generation:3d} | Best: {-np.max(fitness_values):8.4f} | "
                  f"Avg: {-np.mean(fitness_values):8.4f} | Div: {history['diversity'][-1]:.4f}")
        
        # Selection
        if selection_method == 'rank':
            parents = rank_based_selection(population, fitness_values, pop_size)
        elif selection_method == 'sus':
            parents = stochastic_universal_sampling(population, fitness_values, pop_size)
        elif selection_method == 'boltzmann':
            temp = 10.0 * (1 - generation / max_generations)  # Decreasing temperature
            parents = boltzmann_selection(population, fitness_values, pop_size, temperature=max(temp, 0.1))
        else:
            parents = rank_based_selection(population, fitness_values, pop_size)
        
        # Create offspring
        offspring = []
        
        for i in range(0, pop_size, 2):
            parent1 = parents[i]
            parent2 = parents[min(i+1, pop_size-1)]
            
            # Crossover
            if crossover_method == 'sbx':
                child1, child2 = simulated_binary_crossover(parent1, parent2, eta=eta_crossover, bounds=bounds)
            elif crossover_method == 'blx':
                child1, child2 = blx_alpha_crossover(parent1, parent2, alpha=0.5, bounds=bounds)
            elif crossover_method == 'arithmetic':
                child1, child2 = arithmetic_crossover(parent1, parent2, alpha=0.5)
            else:
                child1, child2 = simulated_binary_crossover(parent1, parent2, eta=eta_crossover, bounds=bounds)
            
            # Mutation
            mutation_rate = initial_mutation_rate * (1 - generation / max_generations)
            
            if mutation_method == 'polynomial':
                child1 = polynomial_mutation(child1, mutation_rate, bounds, eta=eta_mutation)
                child2 = polynomial_mutation(child2, mutation_rate, bounds, eta=eta_mutation)
            elif mutation_method == 'adaptive':
                child1 = adaptive_mutation(child1, generation, max_generations, bounds)
                child2 = adaptive_mutation(child2, generation, max_generations, bounds)
            
            offspring.append(child1)
            if len(offspring) < pop_size:
                offspring.append(child2)
        
        new_population = np.array(offspring[:pop_size])
        
        # Elitism
        if elite_size > 0:
            new_fitness = evaluate_fitness(new_population, fitness_function,
                                          num_variables, bounds, bits_per_variable=None,
                                          encoding='real')
            new_population, new_fitness = elitist_replacement(
                population, fitness_values, new_population, new_fitness, elite_size
            )
        
        population = new_population
    
    # Final evaluation
    fitness_values = evaluate_fitness(population, fitness_function,
                                     num_variables, bounds, bits_per_variable=None,
                                     encoding='real')
    
    best_idx = np.argmax(fitness_values)
    best_solution = population[best_idx]
    best_fitness = fitness_values[best_idx]
    
    return best_solution, best_fitness, history

print("✓ Advanced GA implementation complete!")

---

<a name='10'></a>
## 10 - Performance Comparison

Let's compare basic vs advanced GA on a challenging problem!

In [ ]:
print("="*70)
print("PERFORMANCE COMPARISON: Basic vs Advanced GA")
print("Problem: Rastrigin Function (highly multimodal)")
print("="*70)

# Advanced GA
print("\n" + "="*70)
print("Running ADVANCED GA...")
print("="*70)

np.random.seed(42)
best_adv, fitness_adv, history_adv = advanced_genetic_algorithm(
    fitness_function=rastrigin_function,
    num_variables=2,
    bounds=(-5.12, 5.12),
    pop_size=100,
    max_generations=200,
    elite_size=2,
    selection_method='rank',
    crossover_method='sbx',
    mutation_method='polynomial',
    eta_crossover=20,
    eta_mutation=20,
    verbose=True
)

print(f"\n{'='*70}")
print("FINAL RESULTS")
print(f"{'='*70}")
print(f"Best solution: {best_adv}")
print(f"Best objective: {-fitness_adv:.6f}")
print(f"Optimal: [0, 0] with objective 0.0")
print(f"Error: {np.linalg.norm(best_adv):.6f}")

In [ ]:
# Visualize convergence
plot_fitness_evolution(history_adv, title="Advanced GA - Rastrigin Function")

In [ ]:
# Compare different selection methods
print("="*70)
print("Comparing Selection Methods")
print("="*70)

selection_methods = ['rank', 'sus', 'boltzmann']
results = []

for method in selection_methods:
    print(f"\nTesting {method.upper()} selection...")
    
    np.random.seed(123)
    best_sol, best_fit, hist = advanced_genetic_algorithm(
        fitness_function=rastrigin_function,
        num_variables=2,
        bounds=(-5.12, 5.12),
        pop_size=50,
        max_generations=100,
        selection_method=method,
        verbose=False
    )
    
    results.append({
        'method': method,
        'best_fitness': best_fit,
        'history': hist
    })
    
    print(f"  Final objective: {-best_fit:.6f}")

# Plot comparison
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
for result in results:
    plt.plot([-f for f in result['history']['best']], 
             label=result['method'].upper(), linewidth=2)
plt.xlabel('Generation')
plt.ylabel('Best Objective Value')
plt.title('Selection Method Comparison - Convergence')
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')

plt.subplot(1, 2, 2)
for result in results:
    plt.plot(result['history']['diversity'], 
             label=result['method'].upper(), linewidth=2)
plt.xlabel('Generation')
plt.ylabel('Diversity')
plt.title('Selection Method Comparison - Diversity')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

<a name='11'></a>
## 11 - Conclusion

### What You've Learned

Congratulations! You've completed Tutorial 2 on Advanced Genetic Algorithms. You now understand:

✅ **Advanced Selection Methods**
- Rank-based selection (robust to fitness scaling)
- Stochastic Universal Sampling (low variance)
- Boltzmann selection (temperature-based)
- Niching (diversity maintenance)

✅ **Advanced Crossover**
- Arithmetic crossover (conservative)
- BLX-α (exploratory)
- SBX (modern, adaptive)

✅ **Advanced Mutation**
- Polynomial mutation (self-adaptive)
- Adaptive mutation (time-based)
- Self-adaptive mutation (evolving strategies)

✅ **Population Management**
- Elitism (preserving best)
- Steady-state replacement
- Diversity maintenance

✅ **Constraint Handling**
- Penalty functions
- Death penalty
- Repair mechanisms

✅ **Adaptive Parameters**
- Time-based adaptation
- Fitness-based adaptation

### Key Takeaways

1. **No single best method** - different problems need different techniques
2. **Balance is crucial** - exploration vs exploitation, diversity vs convergence
3. **Modern GAs use:**
   - Rank-based or SUS selection
   - SBX crossover with polynomial mutation
   - Elitism (small elite size)
   - Adaptive parameters

4. **Advanced techniques improve:**
   - Robustness
   - Convergence speed
   - Solution quality
   - Ability to handle complex problems

### Recommended Configurations

#### For Continuous Optimization:
```python
- Selection: Rank-based or SUS
- Crossover: SBX (η=20)
- Mutation: Polynomial (η=20)
- Elite size: 2-5% of population
- Population: 50-100
```

#### For Difficult Multimodal Problems:
```python
- Selection: Niching or Boltzmann
- Crossover: BLX-α (more exploration)
- Mutation: Adaptive (start high)
- Maintain diversity actively
- Larger population (100-200)
```

#### For Constrained Problems:
```python
- Penalty functions (tune coefficient)
- Repair mechanisms for bounds
- Death penalty for hard constraints
- Feasibility-preserving operators
```

### Next Steps

In **Tutorial 3**, you will learn:
- Multi-objective optimization (NSGA-II)
- Traveling Salesman Problem (TSP)
- Hybrid algorithms (GA + local search)
- Parallel and distributed GAs
- Advanced constraint handling

### Practice Exercises

1. Implement a GA with decreasing Boltzmann temperature
2. Compare SBX vs BLX-α on different problems
3. Add crowding distance to selection
4. Solve a constrained optimization problem
5. Implement island model GA

---

**Excellent work!** You're now equipped with modern GA techniques used in research and industry! 🎉🚀

**Continue to Tutorial 3 for even more advanced topics!**